# Catalog queries with DuckDB

**Environment:** [uv](https://docs.astral.sh/uv/) project venv from the repo README (`uv venv --python 3.11 .venv`, `uv sync --extra desi --extra dev`). In Jupyter, use kernel **Python (data-lake)** (`uv run python -m ipykernel install --user --name=data-lake --display-name "Python (data-lake)"`).

This notebook demonstrates how to query the HATS-partitioned Parquet catalogs using `CatalogAccessor`.
All queries run in-process via DuckDB – no server required.

In [ ]:
from pathlib import Path
from data_lake.io.catalog import CatalogAccessor, MultiCatalogAccessor
from data_lake.io.crossmatch import CrossmatchAccessor

LAKE_ROOT = Path("/data/lake")  # adjust to your lake root
SURVEY = "des_dr2"

## 1. Open a catalog and inspect its schema

In [ ]:
cat = CatalogAccessor(LAKE_ROOT, SURVEY)
print(cat)
print(f"\nTotal columns: {len(cat.columns)}")
print("First 20 columns:", cat.columns[:20])

## 2. Arbitrary SQL query (column projection)

DuckDB applies column projection natively – only the requested columns are read from disk.

In [ ]:
df = cat.query("""
    SELECT source_id, ra, dec, mag_i, mag_r
    FROM catalog
    WHERE mag_i < 22.0
      AND mag_i - mag_r BETWEEN 0.2 AND 0.8
    ORDER BY mag_i
    LIMIT 1000
""")
df.head()

## 3. Cone search

In [ ]:
# Search within 0.5 deg of a field centre
cone = cat.sources_in_cone(
    ra=150.0,
    dec=2.2,
    radius_deg=0.5,
    columns=["source_id", "ra", "dec", "mag_i"],
)
print(f"Sources in cone: {len(cone)}")
cone.head()

## 4. Return as Astropy Table (for astronomers preferring familiar APIs)

In [ ]:
astropy_table = cat.sources_in_cone(
    ra=150.0, dec=2.2, radius_deg=0.5, fmt="astropy"
)
astropy_table[:5]

## 5. Multi-survey join via shared DuckDB connection

In [ ]:
with MultiCatalogAccessor(LAKE_ROOT, ["des_dr2", "kids_dr4"]) as multi:
    joined = multi.query("""
        SELECT a.source_id AS sid_des,
               b.source_id AS sid_kids,
               a.mag_i    AS mag_i_des,
               b.mag_i    AS mag_i_kids
        FROM des_dr2 AS a
        JOIN kids_dr4 AS b
          ON a.source_id = b.des_source_id   -- assumes crossmatch already applied
        WHERE a.mag_i < 22.0
        LIMIT 500
    """)
joined.head()

## 6. Cross-match query

In [ ]:
with CrossmatchAccessor(LAKE_ROOT, "des_dr2", "kids_dr4") as xm:
    matches = xm.get_matches(source_id_a=12345678, max_sep_arcsec=1.0)
matches

## 7. Tile-level access (retrieve all sources in a HEALPix tile)

In [ ]:
tile_df = cat.sources_in_tile(npix=1234, columns=["source_id", "ra", "dec", "mag_i"])
print(f"Sources in tile 1234: {len(tile_df)}")
tile_df.head()